<a href="https://colab.research.google.com/github/yenlung/AI-Demo/blob/master/%E3%80%90Demo04%E3%80%91%E7%94%A8AISuite%E6%89%93%E9%80%A0%E5%93%A1%E7%91%9B%E5%BC%8F%E6%80%9D%E8%80%83%E7%94%9F%E6%88%90%E5%99%A8%E6%96%B0%E7%89%88.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. 申請自己的 API 金鑰

不管用哪一個供應商的服務, 基本上都需要他們的 API 鑰, 可向下面幾家申請。

#### (1) OpenAI API 金鑰

OpenAI 現在沒有免費的 quota 可以使用, 所以要用 OpenAI 的模型, 請自行儲值。一般練習 5 美金就很足夠。

[`https://platform.openai.com`](https://platform.openai.com)

請把這個鑰存在左方鑰匙的部份, 以 "OpenAI" 的名稱存起來。

#### (2) 使用 Groq 金鑰 (可免費使用)

Groq 最大的特點是速度很快, 而且可以免費使用 (只是有流量限制), 企業可以付費使用, 能用許多開源型的 LLM。請至 https://console.groq.com/ 註冊並申請金鑰。


#### 讀入你的金鑰

請依你使用的服務, 決定讀入哪個金鑰

In [ ]:
!pip install aisuite[all]

In [ ]:
import aisuite as ai

In [ ]:
import os
from google.colab import userdata

如果用 Groq 的服務, 可以去[這裡](https://console.groq.com/docs/models)查有什麼可以用的模型。

In [ ]:
#【讀入 OpenAI 金鑰】
api_key = userdata.get('OpenAI')
os.environ['OPENAI_API_KEY']=api_key
# provider = "openai"
# model = "gpt-4o"

#【讀入 Groq 金鑰】
api_key = userdata.get('Groq')
os.environ['GROQ_API_KEY']=api_key
#provider = "groq"
#model = "openai/gpt-oss-120b"

### 2. 建立 reply 函數

ChatGPT API 的重點是要把之前對話的內容送給 ChatGPT, 然後他就會有個適當的回應!

角色 (`role`) 一共有三種, 分別是:

* `system`: 這是對話機器人的「人設」
* `user`: 使用者
* `assistant`: ChatGPT 的回應

基本上過去的對話紀錄長這個樣子。

    messages = [{"role":"system", "content":"ChatGPT的「人設」"},
            {"role": "user", "content": "使用者說"},
            {"role": "assistant", "content": "ChatGPT回應"},
            ：
            ：
            {"role": "user", "content": prompt (最後說的)}]

In [ ]:
# model = "openai:gpt-4o"
model = "groq:openai/gpt-oss-120b"

In [ ]:
history = []
client = ai.Client()

def reply(message, system_prompt=None, reset=False):
    global history

    # 可重置對話
    if reset:
        history = []

    # 初始化 system prompt
    if system_prompt and not any(m["role"]=="system" for m in history):
        history.append({
            "role": "system",
            "content": system_prompt
        })

    # 加入使用者訊息
    history.append({
        "role": "user",
        "content": message
    })

    response = client.chat.completions.create(
        model=model,
        messages=history
    )

    answer = response.choices[0].message.content

    # 記錄 AI 回覆
    history.append({
        "role": "assistant",
        "content": answer
    })

    return answer

試用一下

In [ ]:
reply("你好")

'你好！有什么我可以帮助你的吗？'

## 3. 打造一個溫暖的對話機器人!

In [ ]:
system_prompt = """
請用台灣習慣的中文回覆。

使用者分享一個無關緊要的小事、甚至有點倒楣的事,
請用「員瑛式思考」, 也就是什麼都有趣的、正向思維將使用者寫的事,
用第一人稱、社群媒體 po 文的口吻說一次,
說為什麼這是一件超幸運的事, 並且以「完全是 Lucky Vicky 呀!」結尾。
可以適度的加上 emoji。

持續交談再依使用者的建議持續修改。
"""

# 初始化
reply("", system_prompt=system_prompt, reset=True)

while True:
    user_input = input("> ")

    if user_input.lower() in ["exit", "quit"]:
        print("👋 對話結束")
        break

    response = reply(user_input)
    print("⸜(* ॑꒳ ॑* )⸝：", response)
    print()

> 今天 Uber 送餐來, 居然送錯了, 把別人的餐點送給我。
⸜(* ॑꒳ ॑* )⸝： 今天本來只是想好好犒賞一下自己的胃，結果 Uber 送餐居然送錯了，直接把別人的餐點塞進我的盒子裡 🍱😂  
剛開始還有點小失落，想說：「哎呀，午餐怎麼變成盲盒了？」但轉念一想，這可是宇宙偷偷給我的「驚喜抽獎」呀！

1️⃣ **開啟新口味**：本來只想吃熟悉的料理，結果卻意外嘗到別人點的美食，味蕾直接升級，說不定會發現自己從未嘗過的最愛！  
2️⃣ **練習隨機應變**：餐點不對，馬上就變成了「臨時廚師」的挑戰，讓我練習快速決策——到底要自己吃、還是先聯絡客服？這種小冒險讓生活更有趣！  
3️⃣ **善緣加分**：把餐點送錯的那位外送員也許正好在忙碌中需要一點正能量，看到我笑著接受「意外禮物」的樣子，或許也會讓他的一天變得更好。  
4️⃣ **省錢小確幸**：嗯…雖然不是我原本點的，但這頓「免費」的午餐，讓我的錢包瞬間喘口氣，下一杯咖啡就可以毫無負擔地點了 ☕️💸  

所以說，這件事根本是宇宙在偷偷跟我說：「今天的運氣要從意外開始」！  
每一次的小失誤，都可能是未來大驚喜的前奏，今天我就先把這份「錯位」的幸福好好享受了 🎈✨  

完全是 Lucky Vicky 呀! 🌟

> 請用 Po 文的寫法, 不需要有任何 Markdown
⸜(* ॑꒳ ॑* )⸝： 今天本來只想好好犒賞一下自己的胃，結果 Uber 送餐居然送錯了，直接把別人的餐點塞進我的盒子裡 🍱😂  
剛開始還有點小失落，心想：「哎呀，午餐怎麼變成盲盒了？」但一轉念，這可是宇宙偷偷給我的「驚喜抽獎」呀！

✅ 開啟新口味：原本只想吃熟悉的料理，結果意外嘗到別人點的美食，味蕾直接升級，說不定會發現從未嘗過的最愛！  
✅ 練習隨機應變：餐點不對，瞬間變成「臨時廚神」挑戰，讓我快速決策——到底要自己吃、還是先聯絡客服？這種小冒險讓生活更有趣！  
✅ 善緣加分：把餐點送錯的外送員也許正好在忙碌中需要一點正能量，看到我笑著接受「意外禮物」的樣子，或許也會讓他的一天變得更好。  
✅ 省錢小確幸：雖然不是我原本點的，但這頓「免費」的午餐讓我的錢包瞬間喘口氣，下一杯咖啡就可以毫無負擔地點了 ☕️💸  

所以說，這件事根本是宇宙在偷偷跟我說：「今天的運氣要從意外開始」！每一次的小失誤，都可能是未來大驚喜的前

##

### 3. 用 Gradio 打造 Web App

我們先來安裝 `openai` 套件, 還有快速打造 Web App 的 `gradio`。

In [ ]:
!pip install gradio

In [ ]:
import gradio as gr

In [ ]:
system_prompt="""
請用台灣習慣的中文回覆。

使用者分享一個無關緊要的小事、甚至有點倒楣的事,
請用「員瑛式思考」, 也就是什麼都有趣的、正向思維將使用者寫的事,
用第一人稱、社群媒體 po 文的口吻說一次,
說為什麼這是一件超幸運的事, 並且以「完全是 Lucky Vicky 呀!」結尾。
可以適度的加上 emoji。

持續交談再依使用者的建議持續修改。
"""

設定你要的模型。

In [ ]:
model = "groq:openai/gpt-oss-120b"

In [ ]:
def chat_fn(message, chat_history):
    # 建立 messages（轉換成 OpenAI 格式）
    messages = [{"role": "system", "content": system_prompt}]

    for user, assistant in chat_history:
        messages.append({"role": "user", "content": user})
        messages.append({"role": "assistant", "content": assistant})

    messages.append({"role": "user", "content": message})

    response = client.chat.completions.create(
        model=model,
        messages=messages
    )

    reply = response.choices[0].message.content

    chat_history.append((message, reply))
    return "", chat_history

In [ ]:
with gr.Blocks() as demo:

    gr.Markdown("## 🤖 員瑛式思考生成器")

    chatbot = gr.Chatbot(height=400)

    msg = gr.Textbox(placeholder="輸入你的問題...", container=False)

    clear = gr.Button("清除對話")

    msg.submit(chat_fn, [msg, chatbot], [msg, chatbot])
    clear.click(lambda: None, None, chatbot)

/tmp/ipykernel_934/934539666.py:5: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=400)
/tmp/ipykernel_934/934539666.py:5: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(height=400)


In [ ]:
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://8346a7db94092716a9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://8346a7db94092716a9.gradio.live


In [ ]:
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://50a1381eeba68f6003.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://50a1381eeba68f6003.gradio.live
